In [6]:
# %%
import pickle
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    auc,
    roc_auc_score
)
import matplotlib.pyplot as plt
import seaborn as sns

from d2c.benchmark.utils import prepare_prediction_df_d2c

In [7]:
# ==============================================================================
# --- 1. SCRIPT CONFIGURATION ---
# ==============================================================================
DATA_DIR = Path('data/causal_dfs/')
FILE_TO_PROCESS = 'causal_dfs_TEST.pkl'
FIGURES_DIR = Path('PR_ROC_FIGURES/')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
# ==============================================================================
# --- 2. DATA LOADING AND PREPARATION (CORRECTED) ---
# ==============================================================================
input_path = DATA_DIR / FILE_TO_PROCESS
print("=" * 60)
print(f"Loading data from: {input_path}")
try:
    with open(input_path, 'rb') as f:
        loaded_data = pickle.load(f)
except FileNotFoundError:
    print(f"FATAL ERROR: File not found at {input_path}. Cannot proceed.")
    exit()

causal_dfs_d2c, true_causal_dfs = loaded_data[-2], loaded_data[-1]

prediction_truth_df = prepare_prediction_df_d2c(causal_dfs_d2c, true_causal_dfs)

Loading data from: data/causal_dfs/causal_dfs_TEST.pkl


In [9]:

# ==============================================================================
# --- 3. ANALYSIS & PLOTTING ---
# ==============================================================================
plt.style.use('seaborn-v0_8-whitegrid')

# Use the y_true that corresponds to this method's aligned predictions
y_true = prediction_truth_df.y_true
y_proba = prediction_truth_df.y_pred

# --- Step 1: Find the PR Break-Even Point and its Threshold ---
precision, recall, pr_thresholds = precision_recall_curve(y_true, y_proba)
pr_auc = auc(recall, precision)

# Find the point where Precision and Recall are closest
# Note: We use precision[:-1] and recall[:-1] to match the length of pr_thresholds
diffs = np.abs(precision[:-1] - recall[:-1])
be_idx = np.argmin(diffs) # be_idx = "break-even index"

# Get the values at this break-even point
be_threshold = pr_thresholds[be_idx]
be_precision = precision[be_idx]
be_recall    = recall[be_idx]

print(f"  PR Break-Even Point (P≈R):")
print(f"    - Threshold: {be_threshold:.4f}")
print(f"    - Precision: {be_precision:.4f}")
print(f"    - Recall:    {be_recall:.4f}")

# --- Step 2: Generate the PR Curve Plot ---
fig_pr, ax_pr = plt.subplots(figsize=(10, 9))

ax_pr.plot(recall, precision, lw=2.5, label=f'TD2C (AP = {pr_auc:.3f})', zorder=5)
ax_pr.scatter(be_recall, be_precision, marker='*', color='purple', s=300, zorder=10, label=f'P≈R Break-Even')
ax_pr.annotate(
    f'P≈R Point\nThresh={be_threshold:.2f}\nP={be_precision:.2f}, R={be_recall:.2f}',
    xy=(be_recall, be_precision),
    xytext=(be_recall + 0.1, be_precision - 0.1),
    arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5),
    bbox=dict(boxstyle="round,pad=0.3", fc="purple", ec="black", lw=0.5, alpha=0.6)
)

# Style the PR plot
no_skill = len(y_true[y_true==1]) / len(y_true)
ax_pr.plot([0, 1], [no_skill, no_skill], linestyle='--', lw=2, color='navy', label=f'No-Skill (AP={no_skill:.3f})')
ax_pr.plot([0, 1], [0, 1], linestyle=':', lw=2, color='gray', label='Precision = Recall (Bisettrice)')
ax_pr.set_xlabel('Recall', fontsize=14); ax_pr.set_ylabel('Precision', fontsize=14)
ax_pr.set_xlim([-0.05, 1.05]); ax_pr.set_ylim([-0.05, 1.05])
ax_pr.legend(loc="best"); ax_pr.grid(True)
ax_pr.set_aspect('equal', adjustable='box')

pr_output_path = FIGURES_DIR / f'pr_curve_BE.png'
fig_pr.savefig(pr_output_path, dpi=300, bbox_inches='tight')
plt.close(fig_pr)
print(f"  > PR Curve saved to: {pr_output_path}")


# --- Step 3: Find the corresponding point on the ROC curve ---
fpr, tpr, roc_thresholds = roc_curve(y_true, y_proba)
roc_auc = roc_auc_score(y_true, y_proba)

# Find the index in the ROC thresholds array that is closest to our break-even threshold
roc_idx_at_be = np.argmin(np.abs(roc_thresholds - be_threshold))
be_fpr = fpr[roc_idx_at_be]
be_tpr = tpr[roc_idx_at_be]

print(f"  Corresponding Point on ROC Curve:")
print(f"    - Using Threshold: {roc_thresholds[roc_idx_at_be]:.4f} (closest to {be_threshold:.4f})")
print(f"    - False Positive Rate: {be_fpr:.4f}")
print(f"    - True Positive Rate:  {be_tpr:.4f}")


# --- Step 4: Generate the ROC Curve Plot ---
fig_roc, ax_roc = plt.subplots(figsize=(10, 9))

ax_roc.plot(fpr, tpr, lw=2.5, label=f'TD2C (AUC = {roc_auc:.3f})', zorder=5)
ax_roc.scatter(be_fpr, be_tpr, marker='*', color='purple', s=300, zorder=10, label=f'Point from PR Break-Even')
ax_roc.annotate(
    f'From PR Break-Even\nThresh={be_threshold:.2f}\nFPR={be_fpr:.2f}, TPR={be_tpr:.2f}',
    xy=(be_fpr, be_tpr),
    xytext=(be_fpr + 0.1, be_tpr - 0.15),
    arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=5),
    bbox=dict(boxstyle="round,pad=0.3", fc="purple", ec="black", lw=0.5, alpha=0.6)
)

# Style the ROC plot
ax_roc.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='No-Skill (AUC=0.50)')
ax_roc.set_xlabel('False Positive Rate', fontsize=14); ax_roc.set_ylabel('True Positive Rate', fontsize=14)
ax_roc.set_xlim([-0.05, 1.05]); ax_roc.set_ylim([-0.05, 1.05])
ax_roc.legend(loc="lower right"); ax_roc.grid(True)
ax_roc.set_aspect('equal', adjustable='box')

roc_output_path = FIGURES_DIR / f'roc_curve_with_BE_point.png'
fig_roc.savefig(roc_output_path, dpi=300, bbox_inches='tight')
plt.close(fig_roc)
print(f"  > ROC Curve saved to: {roc_output_path}")

  PR Break-Even Point (P≈R):
    - Threshold: 0.5480
    - Precision: 0.6574
    - Recall:    0.6567
  > PR Curve saved to: PR_ROC_FIGURES/pr_curve_BE.png
  Corresponding Point on ROC Curve:
    - Using Threshold: 0.5480 (closest to 0.5480)
    - False Positive Rate: 0.0632
    - True Positive Rate:  0.6567
  > ROC Curve saved to: PR_ROC_FIGURES/roc_curve_with_BE_point.png
